# SPLITTING

In [ ]:
from zipper_owntry import Zipp3r

In [ ]:
image_in = '/app/data/data1channel-001.tif'
output_path = '/app/Zipping-pruebas/OUTPUT'

In [ ]:
zipper = Zipp3r(image_in, output_path)

# Create a dictionary containing information on all blocks
zipper.set_block_slices(
    image_shape=[146, 3788, 3788],
    blocksize=[146, 1894, 1894],
    margins=[0,100,100]
)

In [ ]:
# Cut the image according to the block_slices
zipper.split_image()
print('Done!')

# STITCHING TIFF

In [ ]:
import os
import re
import numpy as np
import tifffile

In [ ]:
def load_tiff(file_path):
    """Carga un archivo TIFF como un array numpy."""
    with tifffile.TiffFile(file_path) as tif:
        return tif.asarray()

In [ ]:
def parse_filename(filename):
    """Extrae las coordenadas Z, X, Y desde el nombre del archivo."""
    pattern = r'_block(\d+)x(\d+)x(\d+)\.tif$'
    match = re.search(pattern, filename)
    if match:
        return int(match.group(1)), int(match.group(2)), int(match.group(3))
    else:
        raise ValueError(f"Filename {filename} does not match the expected pattern.")

In [ ]:
def stitch_tiles(tiles, overlap_x, overlap_y):
    """Realiza el stitching de los tiles considerando el overlapping."""

    # Obtener la cantidad de tiles en la dirección Y (filas) y X (columnas)
    n_tiles_y = len(tiles)
    n_tiles_x = len(tiles[0])
    print(f"Cantidad de tiles - Filas (Y): {n_tiles_y}, Columnas (X): {n_tiles_x}")

    # Determinar el tamaño máximo de los tiles en Y y X
    max_tile_y = max(max(tile.shape[1] for tile in row) for row in tiles)
    max_tile_x = max(max(tile.shape[2] for tile in row) for row in tiles)
    min_tile_y = min(min(tile.shape[1] for tile in row) for row in tiles)
    min_tile_x = min(min(tile.shape[2] for tile in row) for row in tiles)
    print(f"Tamaño máximo de los tiles - Alto (Y): {max_tile_y}, Ancho (X): {max_tile_x}")
    print(f"Tamaño minimo de los tiles - Alto (Y): {min_tile_y}, Ancho (X): {min_tile_x}")

    # Calcular el tamaño de la imagen final teniendo en cuenta el solapamiento
    stitched_y = max_tile_y * (n_tiles_y - 2) + min_tile_y * 2 - 2 * overlap_y * (n_tiles_y - 1)
    stitched_x = max_tile_x * (n_tiles_x - 2) + min_tile_x * 2 - 2 * overlap_x * (n_tiles_x - 1)
    print(f"Tamaño de la imagen final - Alto (Y): {stitched_y}, Ancho (X): {stitched_x}")

    # Crear una imagen vacía y un mapa de pesos para manejar el solapamiento
    stitched_image = np.zeros((tiles[0][0].shape[0], stitched_y, stitched_x), dtype=np.float32)
    weight_map = np.zeros_like(stitched_image)

    # Recorrer cada tile para colocarlo en su posición correspondiente en la imagen final
    for i in range(n_tiles_y):
        for j in range(n_tiles_x):
            tile = tiles[i][j]
            tile_z, tile_y, tile_x = tile.shape
            print(f"\nProcesando tile en fila {i}, columna {j}: tamaño (Z, Y, X) = ({tile_z}, {tile_y}, {tile_x})")

            # Calcular el solapamiento para este tile en las direcciones X e Y
            left_overlap = overlap_x if j > 0 else 0
            right_overlap = overlap_x if j < n_tiles_x - 1 else 0
            top_overlap = overlap_y if i > 0 else 0
            bottom_overlap = overlap_y if i < n_tiles_y - 1 else 0

            # Calcular la posición inicial y final del tile en la imagen final
            if n_tiles_y == 2:
                start_y = i * (max_tile_y - top_overlap * 2)
                start_x = j * (max_tile_x - left_overlap * 2)
            else:
                if i == 1:
                    start_y = i * (max_tile_y - top_overlap * 3)
                else:
                    start_y = i * (max_tile_y - top_overlap * 2) - top_overlap
                    
                if j == 1:
                    start_x = j * (max_tile_x - left_overlap * 3)
                else:
                    start_x = j * (max_tile_x - left_overlap * 2) - left_overlap

            end_y = start_y + tile_y
            end_x = start_x + tile_x
            print(f"Inicio X: {start_x}, Fin X: {end_x}")
            print(f"Inicio Y: {start_y}, Fin Y: {end_y}")

            # Asegurarse de que los índices de fin estén dentro del rango de la imagen final
            end_y = min(end_y, stitched_y)
            end_x = min(end_x, stitched_x)

            # Asegurarse de que los índices de inicio estén dentro del rango válido
            start_y = max(start_y, 0)
            start_x = max(start_x, 0)

            # Calcular los tamaños finales del tile para asegurarse de que no se salga del rango
            tile_end_y = min(tile_y, end_y - start_y)
            tile_end_x = min(tile_x, end_x - start_x)

            # Cortar el tile para evitar cualquier superposición no deseada
            tile_cut = tile[:, :tile_end_y, :tile_end_x]

            # Sumar la intensidad del tile a la imagen final en la posición correspondiente
            stitched_image[:, start_y:end_y, start_x:end_x] += tile_cut

            # Actualizar el mapa de pesos
            weight_map[:, start_y:end_y, start_x:end_x] += 1

    # Normalizar la imagen final para evitar saturación en las zonas de solapamiento
    stitched_image /= np.maximum(weight_map, 1)

    # Convertir la imagen final a tipo uint16 para reducir el uso de memoria
    final_image = stitched_image.astype(np.uint16)
    print(f"\nImagen final convertida a tipo uint16 con tamaño: {final_image.shape}")
    return final_image

In [ ]:
def process_directory(directory, overlap_x, overlap_y):
    """Lee todos los archivos TIFF en el directorio y realiza el stitching."""
    tile_dict = {}

    # Recorre todos los archivos en el directorio
    for filename in os.listdir(directory):
        if filename.endswith(".tif"):
            z, x, y = parse_filename(filename)
            file_path = os.path.join(directory, filename)
            tile = load_tiff(file_path)

            if z not in tile_dict:
                tile_dict[z] = []
            tile_dict[z].append((x, y, tile))

    # Ordenar los tiles por sus coordenadas X, Y
    for z in tile_dict:
        tile_dict[z] = sorted(tile_dict[z], key=lambda item: (item[0], item[1]))

    # Determinar dimensiones de la matriz de tiles
    z_levels = sorted(tile_dict.keys())
    stitched_tiles = []
    for z in z_levels:
        # Determinar el tamaño de la matriz de tiles
        max_x = max(x for x, _, _ in tile_dict[z]) + 1
        max_y = max(y for _, y, _ in tile_dict[z]) + 1
        tile_matrix = [[None] * max_y for _ in range(max_x)]

        for x, y, tile in tile_dict[z]:
            tile_matrix[x][y] = tile

        stitched_tiles.append(stitch_tiles(tile_matrix, overlap_x, overlap_y))

    return stitched_tiles

In [ ]:
# Ejemplo de uso
directory = "/app/data/1chanel_tiles"  # Cambia esto por la ruta de tu directorio
overlap_x = 100  # Cambia esto según tu configuración
overlap_y = 100  # Cambia esto según tu configuración

In [ ]:
# Procesa el directorio y realiza el stitching
stitched_images = process_directory(directory, overlap_x, overlap_y)

In [ ]:
# Guardar las imágenes unidas por cada nivel Z como TIFF
for i, stitched_image in enumerate(stitched_images):
    output_path = f'/app/stitched_image_z{overlap_x}aaa.tiff'
    tifffile.imwrite(output_path, stitched_image)

# STITCHING PNG

In [ ]:
import os
import re
import numpy as np
from PIL import Image

In [ ]:
def load_png(file_path):
    """Carga un archivo PNG como un array numpy."""
    with Image.open(file_path) as img:
        return np.array(img)

In [ ]:
def parse_filename(filename):
    """Extrae las coordenadas Z, X, Y desde el nombre del archivo."""
    pattern = r'_block(\d+)x(\d+)x(\d+)\.png$'
    match = re.search(pattern, filename)
    if match:
        return int(match.group(1)), int(match.group(2)), int(match.group(3))
    else:
        raise ValueError(f"Filename {filename} does not match the expected pattern.")

In [ ]:
def stitch_tiles(tiles, overlap_x, overlap_y):
    """Realiza el stitching de los tiles considerando el overlapping."""

    n_tiles_y = len(tiles)
    n_tiles_x = len(tiles[0])
    print(f"Cantidad de tiles - Filas (Y): {n_tiles_y}, Columnas (X): {n_tiles_x}")

    max_tile_y = max(max(tile.shape[0] for tile in row) for row in tiles)
    max_tile_x = max(max(tile.shape[1] for tile in row) for row in tiles)
    min_tile_y = min(min(tile.shape[0] for tile in row) for row in tiles)
    min_tile_x = min(min(tile.shape[1] for tile in row) for row in tiles)
    print(f"Tamaño máximo de los tiles - Alto (Y): {max_tile_y}, Ancho (X): {max_tile_x}")
    print(f"Tamaño mínimo de los tiles - Alto (Y): {min_tile_y}, Ancho (X): {min_tile_x}")

    stitched_y = max_tile_y * (n_tiles_y - 2) + min_tile_y * 2 - 2 * overlap_y * (n_tiles_y - 1)
    stitched_x = max_tile_x * (n_tiles_x - 2) + min_tile_x * 2 - 2 * overlap_x * (n_tiles_x - 1)
    print(f"Tamaño de la imagen final - Alto (Y): {stitched_y}, Ancho (X): {stitched_x}")

    stitched_image = np.zeros((stitched_y, stitched_x, tiles[0][0].shape[2]), dtype=np.float32)
    weight_map = np.zeros_like(stitched_image)

    for i in range(n_tiles_y):
        for j in range(n_tiles_x):
            tile = tiles[i][j]
            tile_y, tile_x, tile_channels = tile.shape
            print(f"\nProcesando tile en fila {i}, columna {j}: tamaño (Y, X, C) = ({tile_y}, {tile_x}, {tile_channels})")

            left_overlap = overlap_x if j > 0 else 0
            right_overlap = overlap_x if j < n_tiles_x - 1 else 0
            top_overlap = overlap_y if i > 0 else 0
            bottom_overlap = overlap_y if i < n_tiles_y - 1 else 0

            if i == 1:
                start_y = i * (max_tile_y - top_overlap * 3)
            else:
                start_y = i * (max_tile_y - top_overlap * 2) - top_overlap
                
            if j == 1:
                start_x = j * (max_tile_x - left_overlap * 3)
            else:
                start_x = j * (max_tile_x - left_overlap * 2) - left_overlap

            end_y = start_y + tile_y
            end_x = start_x + tile_x
            print(f"Inicio X: {start_x}, Fin X: {end_x}")

            end_y = min(end_y, stitched_y)
            end_x = min(end_x, stitched_x)
            start_y = max(start_y, 0)
            start_x = max(start_x, 0)

            tile_end_y = min(tile_y, end_y - start_y)
            tile_end_x = min(tile_x, end_x - start_x)

            tile_cut = tile[:tile_end_y, :tile_end_x, :]

            stitched_image[start_y:end_y, start_x:end_x, :] += tile_cut
            weight_map[start_y:end_y, start_x:end_x, :] += 1

    stitched_image /= np.maximum(weight_map, 1)

    final_image = stitched_image.astype(np.uint8)
    print(f"\nImagen final convertida a tipo uint8 con tamaño: {final_image.shape}")
    return final_image

In [ ]:
def process_directory(directory, overlap_x, overlap_y):
    """Lee todos los archivos PNG en el directorio y realiza el stitching."""
    tile_dict = {}

    for filename in os.listdir(directory):
        if filename.endswith(".png"):
            z, x, y = parse_filename(filename)
            file_path = os.path.join(directory, filename)
            tile = load_png(file_path)

            if z not in tile_dict:
                tile_dict[z] = []
            tile_dict[z].append((x, y, tile))

    for z in tile_dict:
        tile_dict[z] = sorted(tile_dict[z], key=lambda item: (item[0], item[1]))

    z_levels = sorted(tile_dict.keys())
    stitched_tiles = []
    for z in z_levels:
        max_x = max(x for x, _, _ in tile_dict[z]) + 1
        max_y = max(y for _, y, _ in tile_dict[z]) + 1
        tile_matrix = [[None] * max_y for _ in range(max_x)]

        for x, y, tile in tile_dict[z]:
            tile_matrix[x][y] = tile

        stitched_tiles.append(stitch_tiles(tile_matrix, overlap_x, overlap_y))

    return stitched_tiles

In [ ]:
# Ejemplo de uso
directory = "/app/Zipping-pruebas/OUTPUT/blocks"  # Cambia esto por la ruta de tu directorio
overlap_x = 100  # Cambia esto según tu configuración
overlap_y = 100  # Cambia esto según tu configuración

stitched_images = process_directory(directory, overlap_x, overlap_y)

In [ ]:
# Guardar las imágenes unidas por cada nivel Z como PNG
numero = 1
for i, stitched_image in enumerate(stitched_images):
    output_path = f'/app/stitched_image_z{overlap_x}_{numero}.png'
    Image.fromarray(stitched_image).save(output_path)
    numero += 1
    print(f'Archivo guardado en {output_path}')

# PADDING

In [16]:
import os
import numpy as np
import tifffile
import re

In [17]:
def load_tiff(file_path):
    """Carga un archivo TIFF como un array numpy."""
    with tifffile.TiffFile(file_path) as tif:
        return tif.asarray()

In [18]:
def parse_filename(filename):
    """Extrae las coordenadas Z, X, Y desde el nombre del archivo."""
    pattern = r'_block(\d+)x(\d+)x(\d+)\.tif$'
    match = re.search(pattern, filename)
    if match:
        return int(match.group(1)), int(match.group(2)), int(match.group(3))
    else:
        raise ValueError(f"Filename {filename} does not match the expected pattern.")

In [ ]:
def add_padding(tile, max_y, max_x, position):
    """Añade padding a un tile basado en su posición."""
    tile_z, tile_y, tile_x = tile.shape

    # Calcular el padding necesario
    pad_top = pad_bottom = pad_left = pad_right = 0

    if 'top' in position:
        pad_bottom = max_y - tile_y
    elif 'bottom' in position:
        pad_top = max_y - tile_y

    if 'left' in position:
        pad_right = max_x - tile_x
    elif 'right' in position:
        pad_left = max_x - tile_x

    # Aplicar el padding
    padded_tile = np.pad(tile, ((0, 0), (pad_top, pad_bottom), (pad_left, pad_right)), mode='constant')
    
    return padded_tile

In [ ]:
def determine_position(x, y, max_x, max_y):
    """Determina la posición del tile en la imagen completa (esquina, borde, centro)."""
    position = []
    if y == 0:
        position.append('top')
    elif y == max_y - 1:
        position.append('bottom')

    if x == 0:
        position.append('left')
    elif x == max_x - 1:
        position.append('right')

    return position

In [ ]:
def process_tiles_with_padding(directory):
    """Carga todos los tiles, añade padding según su posición y guarda los resultados."""
    tile_dict = {}

    # Recorre todos los archivos en el directorio y los carga en un diccionario
    for filename in os.listdir(directory):
        if filename.endswith(".tif"):
            z, x, y = parse_filename(filename)
            file_path = os.path.join(directory, filename)
            tile = load_tiff(file_path)

            if z not in tile_dict:
                tile_dict[z] = []
            tile_dict[z].append((x, y, tile))

    # Determinar las dimensiones máximas de los tiles
    max_tile_y = max(tile.shape[1] for tiles in tile_dict.values() for _, _, tile in tiles)
    max_tile_x = max(tile.shape[2] for tiles in tile_dict.values() for _, _, tile in tiles)

    print(f"Dimensiones máximas - Alto (Y): {max_tile_y}, Ancho (X): {max_tile_x}")

    padded_directory = os.path.join(directory, "padded_tiles")
    os.makedirs(padded_directory, exist_ok=True)

    for z, tiles in tile_dict.items():
        max_x = max(x for x, _, _ in tiles) + 1
        max_y = max(y for _, y, _ in tiles) + 1

        for x, y, tile in tiles:
            position = determine_position(x, y, max_x, max_y)
            padded_tile = add_padding(tile, max_tile_y, max_tile_x, position)

            # Guardar el tile con padding
            padded_filename = f'_block{z}x{x}x{y}_padded.tif'
            padded_path = os.path.join(padded_directory, padded_filename)
            tifffile.imwrite(padded_path, padded_tile)
            print(f"Guardado {padded_path}")

In [ ]:
# Ejemplo de uso
directory = "/app/Zipping-pruebas/OUTPUT/blocks"  # Cambia esto por la ruta de tu directorio
process_tiles_with_padding(directory)